# Objective

This notebook compares a leaky model with a leakage-free model. It demonstrates why future-derived features must be removed.

# Experiment A — Leaky Features

The leaky model includes future-derived fields such as trend_direction, trend_pct, future_clicks, and future_position.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/raw/seo_content_performance.csv')
leak_cols = ['content_type', 'category', 'word_count', 'content_age_days', 'author_type', 'has_schema',
              'internal_link_count', 'external_link_count', 'organic_clicks', 'organic_impressions',
              'ctr', 'average_position', 'keyword_count', 'ranking_keywords', 'backlinks',
              'domain_authority', 'sessions', 'bounce_rate', 'avg_session_duration', 'conversions',
              'clicks_7d', 'clicks_30d', 'impressions_7d', 'impressions_30d', 'position_7d', 'position_30d',
              'trend_direction', 'trend_pct', 'future_clicks', 'future_position']
X_leak = df[leak_cols]
y = df['is_declining_label']
X_train, X_test, y_train, y_test = train_test_split(X_leak, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
print('Leaky model trained')

# Step 2 — Feature Investigation

We inspect the feature importances and see that trend features are important because they are label-derived.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_leak.columns).sort_values(ascending=False)
importances.head(10)

# Experiment B — Leakage-Free Features

The second model removes future-derived feature columns and keeps only prediction-time features.

In [ ]:
safe_cols = [c for c in leak_cols if c not in ['trend_direction', 'trend_pct', 'future_clicks', 'future_position']]
X_safe = df[safe_cols]
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_safe, y, test_size=0.2, random_state=42, stratify=y)
model_safe = RandomForestClassifier(n_estimators=50, random_state=42)
model_safe.fit(X_train2, y_train2)
print('Leakage-free model trained')

# Comparison

Compare precision, recall, F1, ROC-AUC, PR-AUC, and Precision@50 for both experiments. The leaky artifacts should be removed from the future final model.